In [3]:
import torch
from pytorch_tabnet.tab_model import TabNetClassifier
from ehr_models import *
import ehr_utils
import pandas as pd
# drop mean bayesian
fill_method = 'mean'

file_path = f'data_cleaned/ver-noelectrolyte-logtransform_{fill_method}.csv'

X_train, X_test, Y_train, Y_test = ehr_utils.get_train_test(file_path, train_size=0.8, random_state=42)

tabnet_params = dict(
    n_d = 16,  # 可以理解为用来决定输出的隐藏层神经元个数。n_d越大，拟合能力越强，也容易过拟合
    n_a = 16,   # 可以理解为用来决定下一决策步特征选择的隐藏层神经元个数
    n_steps = 4, # 决策步的个数。可理解为决策树中分裂结点的次数
    gamma = 1.5,  # 决定历史所用特征在当前决策步的特征选择阶段的权重，gamma=1时，表示每个特征在所有决策步中至多仅出现1次
    lambda_sparse = 1e-6,  # 稀疏正则项权重，用来对特征选择阶段的特征稀疏性添加约束,越大则特征选择越稀疏
    optimizer_fn = torch.optim.Adam,
    optimizer_params = dict(lr = 1e-2, weight_decay = 1e-5),
    momentum = 0.95,
    mask_type = "entmax",
    seed = 0
)
 
clf = TabNetClassifier(**tabnet_params)
from sklearn.metrics import accuracy_score
clf.fit(
    X_train.values,Y_train.values,
    eval_set=[(X_test.values, Y_test.values)],
    eval_name=['test'],
    eval_metric=['auc'],
    max_epochs=500,
    patience=50,
    batch_size=4096,
    virtual_batch_size=1024,
)
 
tabnet_y_pred = clf.predict(X_test.values)
tabnet_test_accuracy = accuracy_score(Y_test.values, tabnet_y_pred)
print("TabNet Test Accuracy:", tabnet_test_accuracy)
## 需要测试哪个模型就取消注释对应的行,需要默认参数default=True，需要优化后的参数 默认=false
# default = False
# model = init_adaBoost(default_parm=default)
# model = init_decisionTree(default_parm=default)
# model = init_gaussianNB(default_parm=default)
# model = init_gradientBoosting(default_parm=default)
# model = init_lightGBM(default_parm=default)
# model = init_linearDiscriminantAnalysis(default_parm=default)
# model = init_logisticRegression(default_parm=default)
# model = init_MLPClassifier(default_parm=default)
# model = init_randomForest(default_parm=default)
# model = init_XGBoost(default_parm=default)
# model.fit(X_train, Y_train)
# print("Model training completed.")
# Y_prob, Y_pred = ehr_utils.get_prediction_results(model, X_test)
# Ytrain_prob, Ytrain_pred = ehr_utils.get_prediction_results(model, X_train)
# ehr_utils.plot_roc_pr_curves(Y_train, Ytrain_prob, 'dt')
# ehr_utils.plot_roc_pr_curves(Y_test, Y_prob, 'xgb')
# ehr_utils.eval_model(Y_test, Y_prob, Y_pred)

d:\Cache\Conda\envs\EHR\lib\site-packages\pytorch_tabnet\abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 0.40925 | test_auc: 0.53332 |  0:00:15s
epoch 1  | loss: 0.33943 | test_auc: 0.62386 |  0:00:30s
epoch 2  | loss: 0.336   | test_auc: 0.64577 |  0:00:47s
epoch 3  | loss: 0.3342  | test_auc: 0.65717 |  0:01:02s
epoch 4  | loss: 0.33185 | test_auc: 0.66121 |  0:01:18s
epoch 5  | loss: 0.33028 | test_auc: 0.66921 |  0:01:34s
epoch 6  | loss: 0.32895 | test_auc: 0.66963 |  0:01:49s
epoch 7  | loss: 0.32816 | test_auc: 0.67344 |  0:02:05s
epoch 8  | loss: 0.32799 | test_auc: 0.67321 |  0:02:21s
epoch 9  | loss: 0.32725 | test_auc: 0.67816 |  0:02:38s
epoch 10 | loss: 0.32686 | test_auc: 0.67954 |  0:02:53s
epoch 11 | loss: 0.32639 | test_auc: 0.68089 |  0:03:09s
epoch 12 | loss: 0.3262  | test_auc: 0.67828 |  0:03:26s
epoch 13 | loss: 0.326   | test_auc: 0.68228 |  0:03:42s
epoch 14 | loss: 0.32531 | test_auc: 0.68312 |  0:03:59s
epoch 15 | loss: 0.32515 | test_auc: 0.68236 |  0:04:17s
epoch 16 | loss: 0.32509 | test_auc: 0.68171 |  0:04:34s
epoch 17 | loss: 0.32457 | test

d:\Cache\Conda\envs\EHR\lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


TabNet Test Accuracy: 0.8908051228166355


In [3]:
a=X_train.values

In [ ]:
import pickle
with open(f'trained_models_default/{fill_method}/DecisionTree_0.65.pkl', 'wb') as f:
    pickle.dump(model, f)